# Генерация Витрин (Marts Build)

In [ ]:
import polars as pl
from pathlib import Path

# Вспомогательная функция для записи витрин
def write_mart(plan: pl.LazyFrame, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".tmp")
    if temporary.exists():
        temporary.unlink()
    plan.sink_parquet(
        temporary,
        compression="zstd",
        statistics=True,
        mkdir=True,
    )
    temporary.replace(destination)
    print(f"Wrote: {destination}")

PROJECT_DIR = Path(".").resolve().parents[0]
INPUT_PATH = PROJECT_DIR / "data" / "processed" / "multi_event_clean.parquet"
OUTPUT_DIR = PROJECT_DIR / "data" / "marts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DAY_IN_FIVE_SECOND_BINS = 24 * 60 * 60
EVENT_TYPES = ["listen", "like", "unlike", "dislike", "undislike"]


## Подготовка базового ленивого фрейма (Source)

In [ ]:
def scan_source(path: Path) -> pl.LazyFrame:
    source = pl.scan_parquet(path)
    return (
        source
        .select(
            "uid", "item_id", "timestamp", "is_organic", "event_type", 
            "played_ratio_pct", "track_length_seconds",
        )
        .filter(
            pl.col("uid").is_not_null() & pl.col("item_id").is_not_null() & 
            pl.col("timestamp").is_not_null() & pl.col("is_organic").is_in([0, 1]) & 
            pl.col("event_type").cast(pl.String).is_in(EVENT_TYPES)
        )
        .with_columns(
            (pl.col("timestamp").cast(pl.UInt64) // DAY_IN_FIVE_SECOND_BINS)
            .cast(pl.UInt32)
            .alias("time_period"),
            (pl.col("event_type") == "listen").alias("is_listen"),
            pl.when(pl.col("event_type") == "listen")
            .then(pl.col("played_ratio_pct").clip(0, 100).cast(pl.Float64))
            .otherwise(None).alias("played_ratio_capped_pct"),
            pl.when(pl.col("event_type") == "listen")
            .then(pl.col("track_length_seconds").cast(pl.Float64) * pl.col("played_ratio_pct").clip(0, 100).cast(pl.Float64) / 100)
            .otherwise(None).alias("played_seconds_capped"),
        )
    )

print(f"Input: {INPUT_PATH}")
source = scan_source(INPUT_PATH)


## 1. Витрина `mart_daily_metrics`

In [ ]:
def mart_daily_metrics(source: pl.LazyFrame) -> pl.LazyFrame:
    return (
        source
        .group_by("time_period")
        .agg(
            pl.col("uid").n_unique().alias("dau"),
            pl.col("item_id").n_unique().alias("active_items"),
            pl.len().alias("total_events"),
            pl.col("is_listen").sum().alias("listens"),
            (pl.col("event_type") == "like").sum().alias("likes"),
            (pl.col("event_type") == "unlike").sum().alias("unlikes"),
            (pl.col("event_type") == "dislike").sum().alias("dislikes"),
            (pl.col("event_type") == "undislike").sum().alias("undislikes"),
            ((pl.col("event_type") == "listen") & (pl.col("played_ratio_capped_pct") >= 90)).sum().alias("completed_listens"),
            ((pl.col("event_type") == "listen") & (pl.col("played_ratio_capped_pct") < 10)).sum().alias("short_listens"),
            ((pl.col("event_type") == "listen") & (pl.col("is_organic") == 1)).sum().alias("organic_listens"),
            (pl.col("played_seconds_capped").sum() / 3600).alias("played_hours"),
            pl.col("played_ratio_capped_pct").mean().alias("avg_played_ratio_pct"),
        )
        .with_columns(
            pl.when(pl.col("listens") > 0).then(pl.col("completed_listens") / pl.col("listens")).otherwise(None).alias("completion_rate"),
            pl.when(pl.col("listens") > 0).then(pl.col("short_listens") / pl.col("listens")).otherwise(None).alias("short_listen_rate"),
            pl.when(pl.col("listens") > 0).then(pl.col("organic_listens") / pl.col("listens")).otherwise(None).alias("organic_listen_ratio"),
        )
        .sort("time_period")
    )

write_mart(mart_daily_metrics(source), OUTPUT_DIR / "mart_daily_metrics.parquet")


## 2. Витрина `mart_event_daily`

In [ ]:
def mart_event_daily(source: pl.LazyFrame) -> pl.LazyFrame:
    return (
        source
        .group_by(["time_period", "event_type", "is_organic"])
        .agg(
            pl.len().alias("events"),
            pl.col("uid").n_unique().alias("users"),
            pl.col("item_id").n_unique().alias("items"),
        )
        .with_columns(pl.col("event_type").cast(pl.String))
        .sort(["time_period", "event_type", "is_organic"])
    )

write_mart(mart_event_daily(source), OUTPUT_DIR / "mart_event_daily.parquet")


## 3. Витрина `mart_user_segments`

In [ ]:
def mart_user_segments(source: pl.LazyFrame) -> pl.LazyFrame:
    daily_stats = (
        source
        .group_by(["uid", "time_period"])
        .agg(
            (pl.col("event_type") == "like").sum().alias("daily_total_likes"),
            ((pl.col("event_type") == "like") & (pl.col("is_organic") == 1)).sum().alias("daily_organic_likes"),
            ((pl.col("event_type") == "like") & (pl.col("is_organic") == 0)).sum().alias("daily_algo_likes"),
        )
    )
    global_max = source.select(pl.col("time_period").max().alias("max_period"))
    user_spans = (
        source
        .group_by("uid")
        .agg(pl.col("time_period").min().alias("min_period"))
        .join(global_max, how="cross")
    )
    user_grid = (
        user_spans
        .with_columns(
            pl.int_ranges(pl.col("min_period"), pl.col("max_period") + 1).alias("time_period")
        )
        .explode("time_period")
        .select(pl.col("uid"), pl.col("time_period").cast(pl.UInt32))
    )
    segments = (
        user_grid
        .join(daily_stats, on=["uid", "time_period"], how="left")
        .with_columns(
            pl.col("daily_total_likes").fill_null(0),
            pl.col("daily_organic_likes").fill_null(0),
            pl.col("daily_algo_likes").fill_null(0),
        )
        .sort(["uid", "time_period"])
        .with_columns(
            pl.col("daily_total_likes").cum_sum().over("uid").alias("total_likes"),
            pl.col("daily_organic_likes").cum_sum().over("uid").alias("organic_likes"),
            pl.col("daily_algo_likes").cum_sum().over("uid").alias("algo_likes"),
        )
        .with_columns(
            pl.when(pl.col("total_likes") < 5).then(pl.lit("Cold"))
            .when(pl.col("organic_likes") / pl.col("total_likes") > 0.7).then(pl.lit("Explorer"))
            .when(pl.col("algo_likes") / pl.col("total_likes") > 0.7).then(pl.lit("Passive"))
            .otherwise(pl.lit("Mixed"))
            .alias("segment")
        )
        .select("time_period", "uid", "segment")
        .sort(["time_period", "uid"])
    )
    return segments

write_mart(mart_user_segments(source), OUTPUT_DIR / "mart_user_segments.parquet")


## 4. Витрина `mart_content_health`

In [ ]:
def mart_content_health(source: pl.LazyFrame) -> pl.LazyFrame:
    items = (
        source
        .group_by("item_id")
        .agg(
            pl.col("uid").filter(pl.col("event_type") == "listen").n_unique().alias("listeners"),
            pl.col("is_listen").sum().alias("listens"),
            (pl.col("event_type") == "like").sum().alias("total_likes"),
            (pl.col("event_type") == "dislike").sum().alias("dislikes"),
            ((pl.col("event_type") == "listen") & (pl.col("played_ratio_capped_pct") >= 90)).sum().alias("completed_listens"),
            ((pl.col("event_type") == "listen") & (pl.col("played_ratio_capped_pct") < 10)).sum().alias("short_listens"),
            ((pl.col("event_type") == "listen") & (pl.col("is_organic") == 1)).sum().alias("organic_listens"),
            ((pl.col("event_type") == "listen") & (pl.col("is_organic") == 0)).sum().alias("algo_listens"),
            (pl.col("played_seconds_capped").sum() / 3600).alias("played_hours"),
            pl.col("played_ratio_capped_pct").mean().alias("avg_played_ratio_pct"),
            pl.col("track_length_seconds").filter(pl.col("event_type") == "listen").max().alias("track_length_seconds"),
        )
        .with_columns(
            pl.when(pl.col("listens") > 0).then(pl.col("completed_listens") / pl.col("listens")).otherwise(None).alias("completion_rate"),
            pl.when(pl.col("listens") > 0).then(pl.col("short_listens") / pl.col("listens")).otherwise(None).alias("short_listen_rate"),
            pl.when(pl.col("listens") > 0).then(pl.col("organic_listens") / pl.col("listens")).otherwise(None).alias("organic_ratio"),
        )
        .sort(["total_likes", "item_id"], descending=[True, False])
        .with_row_index("_rank", offset=1)
        .with_columns(
            pl.max_horizontal(pl.lit(1, dtype=pl.UInt32), (pl.len() * 0.01).ceil().cast(pl.UInt32)).alias("_head_end"),
            pl.max_horizontal(pl.lit(1, dtype=pl.UInt32), (pl.len() * 0.20).ceil().cast(pl.UInt32)).alias("_torso_end"),
        )
        .with_columns(
            pl.when(pl.col("_rank") <= pl.col("_head_end")).then(pl.lit("Head"))
            .when(pl.col("_rank") <= pl.col("_torso_end")).then(pl.lit("Torso"))
            .otherwise(pl.lit("Tail"))
            .alias("content_tier")
        )
        .drop("_rank", "_head_end", "_torso_end")
    )
    return items

write_mart(mart_content_health(source), OUTPUT_DIR / "mart_content_health.parquet")


## 5. Витрина `mart_user_general`

In [ ]:
def mart_user_general(source: pl.LazyFrame) -> pl.LazyFrame:
    return (
        source
        .group_by(["time_period", "uid"])
        .agg(
            ((pl.col("event_type") == "like") & (pl.col("is_organic") == 1)).sum().alias("organic_likes"),
            ((pl.col("event_type") == "unlike") & (pl.col("is_organic") == 1)).sum().alias("organic_unlikes"),
            ((pl.col("event_type") == "dislike") & (pl.col("is_organic") == 1)).sum().alias("organic_dislikes"),
            ((pl.col("event_type") == "undislike") & (pl.col("is_organic") == 1)).sum().alias("organic_undislikes"),
            ((pl.col("event_type") == "like") & (pl.col("is_organic") == 0)).sum().alias("algo_likes"),
            ((pl.col("event_type") == "unlike") & (pl.col("is_organic") == 0)).sum().alias("algo_unlikes"),
            ((pl.col("event_type") == "dislike") & (pl.col("is_organic") == 0)).sum().alias("algo_dislikes"),
            ((pl.col("event_type") == "undislike") & (pl.col("is_organic") == 0)).sum().alias("algo_undislikes"),
        )
        .sort(["time_period", "uid"])
    )

write_mart(mart_user_general(source), OUTPUT_DIR / "mart_user_general.parquet")
